In [10]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np

In [ ]:
folder=r'Path'
OFolder=r'Output Folder Path'

In [ ]:
file_list=[]
for (root, dirs, file) in os.walk(folder):
    for f in file:
        if ('.xlsx') in f:

                    file_list.append(f)
file_list

In [13]:
file_link=[]

for i in range(len(file_list)):
     for r,d,f in os.walk(folder):
          for files in f:
               if files == file_list[i]:
                    file_link.append(os.path.join(r,files))

In [14]:
len(file_link)

55

In [ ]:
file_link[1]

In [ ]:
cols=['Brand', 'PartNumber', 'PartTerminologyName',  'PAName', 'Value']
df_Match = pd.DataFrame(columns=cols)
df_Match

In [17]:
file_link[1].split("FULL_")[1].split("_20")[0]

'BBFX_Balkamp'

In [ ]:
for i in range(len(file_link)):
    # workbook=openpyxl.load_workbook(file_link[i])
    BrandName= file_link[i].split("FULL_")[1].split("_20")[0]
    print(i,BrandName)
    df_item=pd.read_excel(file_link[i],sheet_name="Items")
    df_Attr=pd.read_excel(file_link[i],sheet_name="PartAttributes", keep_default_na=False, na_values=[''])
    df_item=df_item.astype(str)
    df_Attr=df_Attr.astype(str)
    df=df_Attr.merge(df_item,how="inner")
    df.rename(columns={'Attribute': "PAName",'PartType':'PartTerminologyName','AttributeValue':'Value'}, inplace=True)
    ListColumns=['Brand', 'PartNumber', 'PartTerminologyName',  'PAName', 'Value']
    df=df[ListColumns]
    df=df.drop_duplicates().reset_index(drop=True)
    df['Brand']=BrandName
    df_Match=pd.concat([df_Match,df])   


In [ ]:
df_Match['Brand'].unique()

In [ ]:
df_Match

In [ ]:
df_Filter=pd.read_excel(r"Active_Part_List_BrandMap.xlsx",sheet_name="BrandMap")
df_Filter = df_Filter[df_Filter['Status'] != "Inactive"]

In [ ]:
merge_DF=pd.merge(df_Match,df_Filter,how="inner",  right_on=["BrandName"], left_on=["Brand"],suffixes=('_match', '_filter'))
merge_DF

In [ ]:
merge_DF=merge_DF[['BrandName', 'PartNumber', 'PartTerminologyName',  'PAName', 'Value']]
merge_DF.rename(columns={'BrandName': 'Brand'}, inplace=True)
merge_DF = merge_DF.dropna(subset=['PartTerminologyName'])
merge_DF

In [24]:
df_Attributes=merge_DF[['Brand', 'PartTerminologyName',  'PAName']].drop_duplicates().reset_index(drop=True)

In [25]:
chunk_size=1000000
# Create a list of DataFrames by splitting the original DataFrame
df_chunks = [merge_DF.iloc[i:i + chunk_size] for i in range(0, len(merge_DF), chunk_size)]

In [26]:
with pd.ExcelWriter(OFolder+'\\'+'PDM_Attribute_List.xlsx') as writer:  # doctest: +SKIP
    for i, chunk in enumerate(df_chunks):
        sheet_name = f"Chunk_{i+1}"  # Naming each sheet dynamically
        chunk.to_excel(writer, sheet_name=sheet_name, index=False)
    df_Attributes.to_excel(writer,index=False, sheet_name='FBG_Attributes')